# Ingesta de datos - Capa Bronze

Este notebook realiza la ingesta de archivos CSV del Banco Mundial hacia la capa **Bronze** del Data Lake, sin aplicar transformaciones.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze-worldbank-ingestion")
    .getOrCreate()
)

current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
RAW_ROOT = PROJECT_ROOT / "raw"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"

BRONZE_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw root: {RAW_ROOT}")
print(f"Bronze root: {BRONZE_ROOT}")

Project root: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation
Raw root: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\raw
Bronze root: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze


In [33]:
import csv
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

INDICATOR_FOLDERS = {
    "pib_per_capita": "PIB per cápita (US$ a precios actuales)",
    "indice_gini": "Índice de Gini",
    "desempleo": "Desempleo, total (% de la fuerza laboral total) (estimación modelada de la OIT)",
    "inflacion": "Inflación, deflactor del PIB (%) anual",
    "exportaciones": "Exportaciones de bienes y servicios (% del PIB)",
    "importaciones": "Importaciones de bienes y servicios (% del PIB)",
    "poblacion": "Población, total",
}

CSV_READ_OPTIONS = {
    "header": False,
    "multiLine": False,
    "encoding": "UTF-8",
    "inferSchema": False,
}


def get_world_bank_schema(source_csv_path: Path) -> StructType:
    with open(source_csv_path, "r", encoding="utf-8-sig", newline="") as file:
        non_empty_lines = [line.strip() for line in file if line.strip()]

    header_line = non_empty_lines[2]
    header = next(csv.reader([header_line]))

    return StructType([
        StructField(col_name, StringType(), True)
        for col_name in header
    ])


def read_world_bank_csv(source_csv_path: Path):
    schema = get_world_bank_schema(source_csv_path)

    df = (
        spark.read
        .options(**CSV_READ_OPTIONS)
        .schema(schema)
        .csv(str(source_csv_path))
    )

    first_col = schema.fieldNames()[0]

    return (
        df.filter(F.col(first_col).isNotNull())
        .filter(F.trim(F.col(first_col)) != "")
        .filter(F.col(first_col).isin("Data Source", "Last Updated Date", "Country Name"))
    )

Indicadores a ingerir: 7
- pib_per_capita: PIB per cápita (US$ a precios actuales)
- indice_gini: Índice de Gini
- desempleo: Desempleo, total (% de la fuerza laboral total) (estimación modelada de la OIT)
- inflacion: Inflación, deflactor del PIB (%) anual
- exportaciones: Exportaciones de bienes y servicios (% del PIB)
- importaciones: Importaciones de bienes y servicios (% del PIB)
- poblacion: Población, total


In [41]:
for indicator_key, folder_name in INDICATOR_FOLDERS.items():
    source_folder = RAW_ROOT / folder_name

    if not source_folder.exists():
        raise FileNotFoundError(f"No existe la carpeta origen para '{indicator_key}': {source_folder}")

    csv_files = sorted(source_folder.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No se encontró CSV en la carpeta: {source_folder}")

    source_csv = csv_files[0]
    target_path = BRONZE_ROOT / indicator_key

    df_raw = read_world_bank_csv(source_csv)

    (
        df_raw.write
        .mode("overwrite")
        .parquet(str(target_path))
    )

    print(f"[OK] {indicator_key} -> {target_path}")

print("Ingesta Bronze completada.")
    

[OK] pib_per_capita -> c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze\pib_per_capita
[OK] indice_gini -> c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze\indice_gini
[OK] desempleo -> c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze\desempleo
[OK] inflacion -> c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze\inflacion
[OK] exportaciones -> c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze\exportaciones
[OK] importaciones -> c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze\importaciones
[OK] poblacion -> c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze\poblacion
Ingesta Bronze completada.
